<a href="https://colab.research.google.com/github/bayuaripermana/bayuaripermana/blob/main/Cleaning_Data_AirBnB_Bali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 — Import Library & Load Data

In [5]:
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()
FILE_PATH = list(uploaded.keys())[0]

df_raw = pd.read_csv(FILE_PATH)
df = df_raw.copy()
df.head()

Saving AirBnB_Bali_All.csv to AirBnB_Bali_All (1).csv


,nama,harga,rating,jumlah_review,link,wilayah
0,Home in Denpasar Selatan - Villa Tulip Sanur,"Rp 6,620,192",New,0,https://www.airbnb.com/rooms/1707098436600237984,Kota Denpasar
1,Villa in Denpasar Timur - 1BR Stylish Villa + ...,"Rp 5,555,555",4.89,19,https://www.airbnb.com/rooms/1652545652776271171,Kota Denpasar
2,Room in Denpasar Utara - Non Smoking Room Jogl...,"Rp 1,073,360",4.88,24,https://www.airbnb.com/rooms/1553501449440607054,Kota Denpasar
3,Villa in Denpasar Timur - Bali Sunrise Escape ...,"Rp 7,500,000",5.0,4,https://www.airbnb.com/rooms/1727345500596205020,Kota Denpasar
4,Shared room in Kecamatan Kuta Utara - 1 Bunk B...,"Rp 771,788",4.89,146,https://www.airbnb.com/rooms/879246225603532906,Kota Denpasar


## Step 2 — Eksplorasi Awal (Data Profiling)
Sebelum membersihkan, kenali dulu masalah datanya.


In [6]:
print("Ukuran data :", df.shape)
print("\nTipe Data:")
print(df.dtypes)
print("\nMissing value per colom:")
print(df.isna().sum())
print("\nBaris duplikat (seluruh kolom):", df.duplicated().sum())
print("Link Duplikat:", df.duplicated(subset='link').sum())
print("\nNilai unik rating (teratas):")
print(df['rating'].value_counts(dropna=False).head())
print("\nJumlah listing per wilayah:")
print(df['wilayah'].value_counts())


Ukuran data : (1235, 6)

Tipe Data:
nama             object
harga            object
rating           object
jumlah_review     int64
link             object
wilayah          object
dtype: object

Missing value per colom:
nama               0
harga              0
rating           133
jumlah_review      0
link               0
wilayah            0
dtype: int64

Baris duplikat (seluruh kolom): 0
Link Duplikat: 177

Nilai unik rating (teratas):
rating
5.0     340
New     209
NaN     133
4.96     34
4.97     34
Name: count, dtype: int64

Jumlah listing per wilayah:
wilayah
Kabupaten Gianyar       239
Kabupaten Karangasem    223
Kabupaten Bangli        222
Kabupaten Jembrana      219
Kabupaten Klungkung     126
Kabupaten Buleleng       64
Kota Denpasar            63
Kabupaten Badung         49
Kabupaten Tabanan        30
Name: count, dtype: int64


## Step 3 — Rapikan Teks

In [8]:
for col in ['nama', 'wilayah', 'link']:
  df[col] = (df[col].astype(str)
  .str.replace(r'\s+',' ', regex=True)
  .str.strip())

  df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
  df[['nama']].head()

## Step 4 — Ekstrak ID Listing & Hapus Duplikat
URL Airbnb berisi ID unik (`/rooms/1707098436600237984`). ID inilah kunci yang paling tepat untuk mendeteksi duplikat.

In [9]:
# Ekstrak ID unik dari link (disimpan sebagai teks agar tidak terpotong jadi notasi ilmiah)
df['id_listing'] = df['link'].str.extract(r'rooms/(\d+)')[0]
print("ID kosong:", df['id_listing'].isna().sum())

n_before = len(df)
print("Listing unik:", df['id_listing'].nunique(), "dari", n_before, "baris")

# Hapus duplikat berdasarkan ID, simpan kemunculan pertama
df = df.drop_duplicates(subset='id_listing', keep='first').reset_index(drop=True)
print(f"Baris dihapus: {n_before - len(df)} | Sisa: {len(df)}")

ID kosong: 0
Listing unik: 1058 dari 1235 baris
Baris dihapus: 177 | Sisa: 1058


## Step 5 — Bersihkan Kolom `harga`

In [10]:
df['harga'] = (df['harga'].astype(str)
                          .str.replace(r'[^\d]', '', regex=True)   # buang 'Rp', spasi, koma
                          .astype('int64'))

# Kolom bantu: harga dalam jutaan rupiah, lebih enak dibaca di grafik
df['harga_juta'] = (df['harga'] / 1_000_000).round(2)
df['harga'].describe().round(0)

,harga
count,1058.0
mean,12093792.0
std,12896298.0
min,654500.0
25%,3901400.0
50%,7751375.0
75%,15668452.0
max,127155275.0


## Step 6 — Bersihkan Kolom `rating`
Ada tiga kondisi: angka (mis. `4.89`), teks `"New"`, dan kosong (NaN). Semuanya berarti *belum ada rating yang bisa dihitung*, jadi kita buat kolom status agar informasinya tidak hilang.

In [11]:
# Cek hubungan rating kosong/New dengan jumlah review
print(pd.crosstab(df['rating'].isna() | (df['rating'] == 'New'), df['jumlah_review'] == 0))

# Status rating (dibuat SEBELUM rating dikonversi ke angka)
df['status_rating'] = np.where(df['rating'].isna(), 'Tidak ada data',
                        np.where(df['rating'] == 'New', 'Listing baru', 'Sudah ada rating'))

# Konversi ke numerik: 'New' otomatis jadi NaN
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Validasi rentang rating Airbnb (0-5)
print("Rating di luar 0-5:", ((df['rating'] < 0) | (df['rating'] > 5)).sum())
df['status_rating'].value_counts()

jumlah_review  False  True 
rating                     
False            762      0
True               0    296
Rating di luar 0-5: 0


,count
status_rating,
Sudah ada rating,762
Listing baru,168
Tidak ada data,128


## Step 7 — Pecah Kolom `nama` Menjadi Fitur Baru
Pola: `"<tipe_properti> in <kecamatan> - <judul>"`.

In [12]:
ekstrak = df['nama'].str.extract(r'^(.*?) in (.*?) - (.*)$')
df['tipe_properti'] = ekstrak[0].str.strip()
df['kecamatan']     = ekstrak[1].str.strip()
df['judul_listing'] = ekstrak[2].str.strip()

print("Gagal diekstrak:", ekstrak.isna().any(axis=1).sum())

# Standarisasi kategori
df['kecamatan']     = df['kecamatan'].str.replace(r'^Kecamatan\s+', '', regex=True)  # 'Kecamatan Ubud' -> 'Ubud'
df['tipe_properti'] = df['tipe_properti'].replace({'Rooms': 'Room', 'Place to stay': 'Lainnya'})

# Gabungkan tipe yang sangat jarang muncul (< 1%) menjadi 'Lainnya' agar grafik tidak berantakan
vc = df['tipe_properti'].value_counts(normalize=True)
langka = vc[vc < 0.01].index
df['tipe_properti'] = df['tipe_properti'].replace(dict.fromkeys(langka, 'Lainnya'))

print(df['tipe_properti'].value_counts())
print("\nJumlah kecamatan unik:", df['kecamatan'].nunique())

Gagal diekstrak: 0
tipe_properti
Home          305
Villa         272
Room          180
Lainnya        88
Cabin          64
Guesthouse     56
Treehouse      37
Tiny home      17
Hotel          15
Bungalow       13
Hut            11
Name: count, dtype: int64

Jumlah kecamatan unik: 88


In [13]:
df = df.rename(columns={'wilayah': 'wilayah_pencarian'})

## Step 8 — Deteksi Outlier
Harga sangat tinggi mungkin properti mewah yang valid, jadi **ditandai (flag), bukan dihapus**.

In [14]:
q1, q3 = df['harga'].quantile([0.25, 0.75])
iqr = q3 - q1
batas_atas = q3 + 1.5 * iqr

df['is_outlier_harga'] = df['harga'] > batas_atas
print(f"Batas atas outlier: Rp {batas_atas:,.0f}")
print("Jumlah outlier harga:", df['is_outlier_harga'].sum())
df.loc[df['is_outlier_harga'], ['judul_listing', 'harga']].sort_values('harga', ascending=False).head()

Batas atas outlier: Rp 33,319,031
Jumlah outlier harga: 68


,judul_listing,harga
698,Luxury 5BR sea view Villa & Private Tennis Court,127155275
983,Oceanfront Villa | Pool | Breakfast | 6 Staff,104945980
254,Jungle 9BR Ubud Paradise with spa!,103667937
981,Stella Maris Oceanfront Retreat,95750000
879,"Oceanview 4br Villa,Infinity Pool in East Bali",74867013


## Step 9 — Feature Engineering (Kolom Turunan untuk Visualisasi)
Kolom kategori membuat dashboard lebih mudah dibaca (filter, slicer, legend).

In [15]:
# 1. Kategori harga berdasarkan kuartil
df['kategori_harga'] = pd.qcut(df['harga'], q=4,
                               labels=['Ekonomis', 'Menengah', 'Mahal', 'Premium'])

# 2. Kategori rating
df['kategori_rating'] = pd.cut(df['rating'], bins=[0, 4.5, 4.8, 5.0],
                               labels=['Cukup (<4.5)', 'Bagus (4.5-4.8)', 'Sangat Bagus (>4.8)'],
                               include_lowest=True)
df['kategori_rating'] = df['kategori_rating'].cat.add_categories('Belum ada rating').fillna('Belum ada rating')

# 3. Kategori popularitas berdasarkan jumlah review
df['kategori_review'] = pd.cut(df['jumlah_review'], bins=[-1, 0, 10, 50, np.inf],
                               labels=['0 review', '1-10', '11-50', '>50'])

# 4. Fitur dari judul listing (kata kunci)
judul = df['judul_listing'].str.lower()
df['ada_pool']   = judul.str.contains(r'pool|kolam renang', regex=True)
df['ada_view']   = judul.str.contains(r'view|sunrise|sunset|ocean|beach|sea|rice|volcano|mountain', regex=True)
df['ada_diskon'] = judul.str.contains(r'\d+%\s*off|promo|discount|diskon', regex=True)

df[['tipe_properti','kecamatan','harga','kategori_harga','kategori_rating','kategori_review','ada_pool','ada_view']].head()

,tipe_properti,kecamatan,harga,kategori_harga,kategori_rating,kategori_review,ada_pool,ada_view
0,Home,Denpasar Selatan,6620192,Menengah,Belum ada rating,0 review,False,False
1,Villa,Denpasar Timur,5555555,Menengah,Sangat Bagus (>4.8),11-50,True,False
2,Room,Denpasar Utara,1073360,Ekonomis,Sangat Bagus (>4.8),11-50,False,False
3,Villa,Denpasar Timur,7500000,Menengah,Sangat Bagus (>4.8),1-10,False,True
4,Lainnya,Kuta Utara,771788,Ekonomis,Sangat Bagus (>4.8),>50,False,True


## Step 10 — Validasi Akhir & Export

In [16]:
# Susun urutan kolom
kolom_akhir = ['id_listing', 'judul_listing', 'tipe_properti', 'kecamatan', 'wilayah_pencarian',
               'harga', 'harga_juta', 'kategori_harga', 'is_outlier_harga',
               'rating', 'status_rating', 'kategori_rating',
               'jumlah_review', 'kategori_review',
               'ada_pool', 'ada_view', 'ada_diskon', 'link']
df_clean = df[kolom_akhir]

# Checklist validasi
assert df_clean['id_listing'].is_unique,           "Masih ada ID duplikat"
assert df_clean['harga'].gt(0).all(),             "Ada harga <= 0"
assert df_clean['rating'].dropna().between(0, 5).all(), "Rating di luar 0-5"
assert df_clean[['tipe_properti','kecamatan','judul_listing']].notna().all().all(), "Ada nilai kosong di kolom hasil ekstraksi"
print("✅ Semua validasi lolos")

print("\nBentuk akhir:", df_clean.shape)
print("\nMissing value tersisa:")
print(df_clean.isna().sum()[lambda s: s > 0])
df_clean.info()

✅ Semua validasi lolos

Bentuk akhir: (1058, 18)

Missing value tersisa:
rating    296
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058 entries, 0 to 1057
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   id_listing         1058 non-null   object  
 1   judul_listing      1058 non-null   object  
 2   tipe_properti      1058 non-null   object  
 3   kecamatan          1058 non-null   object  
 4   wilayah_pencarian  1058 non-null   object  
 5   harga              1058 non-null   int64   
 6   harga_juta         1058 non-null   float64 
 7   kategori_harga     1058 non-null   category
 8   is_outlier_harga   1058 non-null   bool    
 9   rating             762 non-null    float64 
 10  status_rating      1058 non-null   object  
 11  kategori_rating    1058 non-null   category
 12  jumlah_review      1058 non-null   int64   
 13  kategori_review    1058 non-null   category
 14  ada_

In [17]:
# Export - utf-8-sig agar karakter khusus terbaca benar di Excel/Power BI
df_clean.to_csv('airbnb_bali_clean.csv', index=False, encoding='utf-8-sig')
files.download('airbnb_bali_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>